In [3]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import re
import time

# Base URL for the website
BASE_URL = "https://sumodb.sumogames.de/"

# Parameters to get all desired columns on the Banzuke page
# These are taken from the URL you provided
BANZUKE_PAGE_PARAMS = "&heya=-1&shusshin=-1&h=on&sh=on&bd=on&hd=on&su=on&w=on&hr=on&ho=on&ch=on&cs=on&cr=on&spr=on&sps=on&snr=on&sns=on"

# Standard headers to mimic a browser visit
HEADERS = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'
}

def get_tournament_ids(start_year_month_str="199107"):
    """
    Fetches the Basho.aspx page and extracts tournament IDs (YYYYMM)
    from the specified start_year_month.
    """
    tournament_ids = []
    basho_list_url = f"{BASE_URL}Basho.aspx"
    print(f"Fetching tournament list from: {basho_list_url}")

    try:
        response = requests.get(basho_list_url, headers=HEADERS, timeout=10)
        response.raise_for_status()
    except requests.exceptions.RequestException as e:
        print(f"Error fetching tournament list: {e}")
        return tournament_ids

    soup = BeautifulSoup(response.content, 'html.parser')
    
    content_div = soup.find('div', class_='simplecontent')
    if not content_div:
        print("Could not find the main content div ('div.simplecontent').")
        return tournament_ids

    data_table = content_div.find('table') 
    if not data_table:
        print("Could not find the data table within 'div.simplecontent'.")
        return tournament_ids
    
    links = data_table.select("a[href*='Banzuke.aspx?b=']")
    
    if not links:
        print(f"No tournament links found in the table with selector a[href*='Banzuke.aspx?b=']")
        return tournament_ids

    start_year_month_int = int(start_year_month_str)

    for link in links:
        href = link.get('href')
        if href:
            match = re.search(r'b=(\d{6})', href)
            if match:
                b_value_str = match.group(1)
                b_value_int = int(b_value_str)
                if b_value_int >= start_year_month_int:
                    tournament_ids.append(b_value_str)
    
    unique_sorted_ids = sorted(list(set(tournament_ids)))
    print(f"Found {len(unique_sorted_ids)} tournaments from {start_year_month_str} onwards.")
    return unique_sorted_ids

def scrape_banzuke_for_tournament(tournament_id):
    """
    Scrapes wrestler data from the Banzuke page (with all extra columns)
    for a given tournament_id (YYYYMM).
    """
    # Construct URL with all parameters for extra data
    banzuke_url = f"{BASE_URL}Banzuke.aspx?b={tournament_id}{BANZUKE_PAGE_PARAMS}"
    print(f"Scraping Banzuke for tournament: {tournament_id} from {banzuke_url}")
    
    try:
        response = requests.get(banzuke_url, headers=HEADERS, timeout=15) # Increased timeout slightly
        response.raise_for_status() 
    except requests.exceptions.HTTPError as http_err:
        print(f"HTTP error occurred while fetching banzuke for {tournament_id}: {http_err} (URL: {banzuke_url})")
        if response.status_code == 403:
            print(f"IMPORTANT: Received a 403 Forbidden error for {tournament_id}. "
                  "The server is likely rate-limiting. "
                  "Please increase the 'delay_seconds' in the 'scrape_all_banzuke_data' call and try again later.")
        return []
    except requests.exceptions.RequestException as e:
        print(f"Request error fetching banzuke for {tournament_id}: {e} (URL: {banzuke_url})")
        return []

    soup = BeautifulSoup(response.content, 'html.parser')
    wrestlers_data = []
    
    banzuke_tables = soup.find_all('table', class_='banzuke')

    if not banzuke_tables:
        print(f"No banzuke tables found for tournament {tournament_id} at {banzuke_url}")
        return wrestlers_data

    for table in banzuke_tables:
        rows = table.find_all('tr')
        for row in rows:
            # Basic check for essential cells
            rank_cell = row.find('td', class_='rank')
            shikona_cell = row.find('td', class_='shikona')
            if not (rank_cell and shikona_cell):
                continue

            rank = rank_cell.get_text(strip=True)
            shikona_primary = ""
            wrestler_id = None
            shikona_anchor = shikona_cell.find('a')
            if shikona_anchor:
                shikona_primary = shikona_anchor.get_text(strip=True)
                if shikona_anchor.has_attr('href'):
                    href = shikona_anchor['href']
                    id_match = re.search(r'r=(\d+)', href)
                    if id_match:
                        wrestler_id = id_match.group(1)
            else:
                shikona_primary = shikona_cell.get_text(separator=' ', strip=True)

            # Helper to get text from a cell by class name
            def get_cell_text(class_name):
                cell = row.find('td', class_=class_name)
                return cell.get_text(strip=True) if cell else ""

            heya = get_cell_text('heya')
            origin = get_cell_text('shusshin') # 'su=on' ensures this
            hw = get_cell_text('htwt') # 'h=on', 'w=on' ensure this
            birthday_age = get_cell_text('birth') # 'bd=on' ensures this
            prev_rank_status = get_cell_text('prev_rank') # 'spr=on'
            prev_basho_result = get_cell_text('prev_score') # 'sps=on'

            # New fields based on extra parameters
            heya_debut = get_cell_text('debut') # 'hd=on'
            highest_rank = get_cell_text('hrank') # 'hr=on'
            honbasho_appearances = get_cell_text('basho') # 'ho=on'
            career_record = get_cell_text('sum') # 'cr=on'
            
            # Fields with class 'results_s' - Makuuchi achievements and Current Streak
            # These appear in a specific order if all relevant params (ch=on, cs=on) are set
            all_results_s_cells = row.find_all('td', class_='results_s')
            makuuchi_achievements = ""
            current_streak = ""

            # Based on observed structure: Makuuchi achievements is the first 'results_s', Streak is the second.
            # This relies on the fixed column order produced by the specific URL parameters.
            # For Asahifuji 199107 with all params:
            # <td class="sum">...</td> (Career)
            # <td class="results_s">Y4 S1 K2 G1</td> (Makuuchi Achievements)
            # <td class="prev_rank">...</td>
            # <td class="prev_score">...</td>
            # <td class="results_s">2W</td> (Streak)
            # The relative order should hold.
            
            # Find the 'sum' cell (Career Record)
            sum_cell = row.find('td', class_='sum')
            streak_cell_after_prev_score = None

            if sum_cell:
                # Makuuchi Achievements cell should be the next 'results_s' after 'sum'
                next_sibling = sum_cell.find_next_sibling('td')
                if next_sibling and next_sibling.has_attr('class') and 'results_s' in next_sibling['class']:
                    makuuchi_achievements = next_sibling.get_text(strip=True)
            
            # Streak cell is the 'results_s' after 'prev_score'
            prev_score_cell_found = row.find('td', class_='prev_score')
            if prev_score_cell_found:
                next_sibling = prev_score_cell_found.find_next_sibling('td')
                if next_sibling and next_sibling.has_attr('class') and 'results_s' in next_sibling['class']:
                     current_streak = next_sibling.get_text(strip=True)

            if rank and shikona_primary:
                wrestlers_data.append({
                    'tournament_id': tournament_id,
                    'wrestler_id': wrestler_id,
                    'rank': rank,
                    'shikona': shikona_primary,
                    'heya': heya,
                    'origin': origin,
                    'hw': hw,
                    'birthday_age': birthday_age,
                    'heya_debut': heya_debut,
                    'highest_rank': highest_rank,
                    'honbasho_appearances': honbasho_appearances,
                    'career_record': career_record,
                    'makuuchi_achievements': makuuchi_achievements,
                    'prev_rank_status': prev_rank_status,
                    'prev_basho_result': prev_basho_result,
                    'current_streak': current_streak
                })
    return wrestlers_data

def scrape_all_banzuke_data(start_year_month="199107", max_tournaments=None, delay_seconds=3):
    """
    Main function to scrape all Banzuke data from a start date.
    Adjust delay_seconds if you encounter 403 errors.
    """
    print(f"Using a delay of {delay_seconds} seconds between tournament page requests.")
    tournament_ids = get_tournament_ids(start_year_month_str=start_year_month)
    
    if not tournament_ids:
        print("No tournament IDs found to scrape.")
        return pd.DataFrame()

    if max_tournaments is not None:
        tournament_ids = tournament_ids[:max_tournaments]
        print(f"Limiting scraping to the first {max_tournaments} tournaments found.")

    all_data = []
    for i, tour_id in enumerate(tournament_ids):
        print(f"Processing tournament {i+1}/{len(tournament_ids)}: {tour_id}")
        data_for_tournament = scrape_banzuke_for_tournament(tour_id)
        if not data_for_tournament and tour_id : 
             print(f"No data returned or error occurred for {tour_id}.")
        all_data.extend(data_for_tournament)
        
        if i < len(tournament_ids) - 1:
            print(f"Waiting for {delay_seconds} seconds before next request...")
            time.sleep(delay_seconds)
            
    df = pd.DataFrame(all_data)
    return df

# --- Main execution ---
if __name__ == '__main__':
    start_basho = "199107" 
    
    custom_delay = 3 # Seconds. Increase if you get 403 errors (e.g., 5, 7, 10 or more)

    print(f"Starting sumo Banzuke scraping from {start_basho} with a {custom_delay}-second delay between pages.")
    
    # For a quick test (e.g., first 2 tournaments):
    banzuke_df = scrape_all_banzuke_data(start_year_month=start_basho, max_tournaments=2, delay_seconds=custom_delay)
    
    # To scrape all tournaments from July 1991 (this will take a significant amount of time):
    # banzuke_df = scrape_all_banzuke_data(start_year_month=start_basho, delay_seconds=custom_delay)
    
    if not banzuke_df.empty:
        print("\nScraping complete.")
        print(f"Total wrestlers' Banzuke entries scraped: {len(banzuke_df)}")
        pd.set_option('display.max_columns', None) # Show all columns
        print("First 5 entries with all columns:")
        print(banzuke_df.head())
        
        # Optional: Save to CSV
        # output_filename = f"sumo_banzuke_data_DETAILED_{start_basho}_to_latest.csv"
        # banzuke_df.to_csv(output_filename, index=False)
        # print(f"\nData saved to {output_filename}")
    else:
        print("No data was scraped. Check for errors or if max_tournaments was too low for results.")

Starting sumo Banzuke scraping from 199107 with a 3-second delay between pages.
Using a delay of 3 seconds between tournament page requests.
Fetching tournament list from: https://sumodb.sumogames.de/Basho.aspx
Found 202 tournaments from 199107 onwards.
Limiting scraping to the first 2 tournaments found.
Processing tournament 1/2: 199107
Scraping Banzuke for tournament: 199107 from https://sumodb.sumogames.de/Banzuke.aspx?b=199107&heya=-1&shusshin=-1&h=on&sh=on&bd=on&hd=on&su=on&w=on&hr=on&ho=on&ch=on&cs=on&cr=on&spr=on&sps=on&snr=on&sns=on
No data returned or error occurred for 199107.
Waiting for 3 seconds before next request...
Processing tournament 2/2: 199109
Scraping Banzuke for tournament: 199109 from https://sumodb.sumogames.de/Banzuke.aspx?b=199109&heya=-1&shusshin=-1&h=on&sh=on&bd=on&hd=on&su=on&w=on&hr=on&ho=on&ch=on&cs=on&cr=on&spr=on&sps=on&snr=on&sns=on
No data returned or error occurred for 199109.
No data was scraped. Check for errors or if max_tournaments was too low f